# 11. End-to-End ML Project - Titanic**Topic:** Complete pipeline from raw data to final test evaluation. The capstone notebook.**Dataset:** Titanic (seaborn).**What you will do:**1. Load and explore2. Clean and engineer features3. Compare 5 models4. Tune the best one with GridSearchCV5. Evaluate on a held-out test set6. Inspect feature importance and predictions

In [ ]:
# Run this cell first if running locally. In Google Colab everything is pre-installed.# !pip install -q numpy pandas scikit-learn matplotlib seaborn scipy xgboost lightgbm

In [ ]:
import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snssns.set_style("whitegrid")plt.rcParams["figure.figsize"] = (10, 6)np.random.seed(42)

## 1. Load and explore

In [ ]:
df = sns.load_dataset("titanic")print(df.shape)df.head()

In [ ]:
print("Missing values:")print(df.isnull().sum().sort_values(ascending=False).head())print()print("Class balance:")print(df["survived"].value_counts(normalize=True).round(3))

## 2. Clean and engineer

In [ ]:
df = df[["survived", "pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]].copy()df["family_size"] = df["sibsp"] + df["parch"] + 1df["is_alone"]    = (df["family_size"] == 1).astype(int)df["age"].fillna(df["age"].median(), inplace=True)df["embarked"].fillna(df["embarked"].mode()[0], inplace=True)df = pd.get_dummies(df, columns=["sex", "embarked"], drop_first=True)df.head()

## 3. Split

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCVfrom sklearn.preprocessing import StandardScalerfrom sklearn.pipeline import PipelineX = df.drop(columns=["survived"])y = df["survived"]X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)print(X_train.shape, X_test.shape)

## 4. Compare 5 models

In [ ]:
from sklearn.linear_model import LogisticRegressionfrom sklearn.tree import DecisionTreeClassifierfrom sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifierfrom sklearn.neighbors import KNeighborsClassifiermodels = {    "Logistic Regression":  LogisticRegression(max_iter=1000),    "Decision Tree":        DecisionTreeClassifier(max_depth=5, random_state=42),    "Random Forest":        RandomForestClassifier(n_estimators=300, random_state=42),    "Gradient Boosting":    GradientBoostingClassifier(random_state=42),    "KNN":                  KNeighborsClassifier(n_neighbors=7),}for name, m in models.items():    pipe = Pipeline([("scaler", StandardScaler()), ("model", m)])    s = cross_val_score(pipe, X_train, y_train, cv=5, scoring="accuracy")    print(f"{name:25s}  CV acc = {s.mean():.3f} (+/- {s.std():.3f})")

## 5. Tune Gradient Boosting

In [ ]:
pipe = Pipeline([    ("scaler", StandardScaler()),    ("model",  GradientBoostingClassifier(random_state=42)),])param_grid = {    "model__n_estimators":  [100, 200, 300],    "model__learning_rate": [0.05, 0.1, 0.2],    "model__max_depth":     [2, 3, 4],}grid = GridSearchCV(pipe, param_grid, cv=5, scoring="accuracy", n_jobs=-1)grid.fit(X_train, y_train)print("Best params:", grid.best_params_)print(f"Best CV acc: {grid.best_score_:.3f}")

## 6. Final test evaluation

In [ ]:
from sklearn.metrics import (    accuracy_score, classification_report, confusion_matrix, roc_auc_score)best = grid.best_estimator_y_pred  = best.predict(X_test)y_proba = best.predict_proba(X_test)[:, 1]print(f"Test accuracy: {accuracy_score(y_test, y_pred):.3f}")print(f"Test ROC-AUC:  {roc_auc_score(y_test, y_proba):.3f}")print()print(classification_report(y_test, y_pred, target_names=["Died", "Survived"]))

In [ ]:
cm = confusion_matrix(y_test, y_pred)sns.heatmap(cm, annot=True, fmt="d", cmap="Greys",            xticklabels=["Died", "Survived"], yticklabels=["Died", "Survived"])plt.xlabel("Predicted")plt.ylabel("Actual")plt.title("Final Test Set Confusion Matrix")plt.show()

## 7. Feature importance

In [ ]:
model = best.named_steps["model"]imp = pd.Series(model.feature_importances_, index=X.columns).sort_values()plt.figure(figsize=(8, 5))imp.plot.barh(color="gray")plt.title("Gradient Boosting Feature Importance")plt.show()

## 8. Inspect predictions

In [ ]:
sample = X_test.head(10).copy()sample["actual"]    = y_test.head(10).valuessample["predicted"] = best.predict(sample.drop(columns=["actual"]))sample["prob_survived"] = best.predict_proba(sample.drop(columns=["actual", "predicted"]))[:, 1].round(3)sample[["actual", "predicted", "prob_survived"]]

## 9. Next steps1. **Try XGBoost / LightGBM** - often beats GradientBoosting.2. **Extract title from passenger name** (load raw Kaggle CSV) for a better feature.3. **Save the model** with `joblib.dump(best, "titanic_model.pkl")`.4. **Deploy** the model behind a Flask or FastAPI endpoint.5. **Try a different problem** - House Prices, IMDB sentiment, MNIST digits.